# von Neumann Derivation - Appendix A

The following is a standalone derivation of the forward Euler stability constraint for the one-group neutron diffusion equation, supporting Section 6 of the main notebook.

### Quick (but actually not so quick) aside: where does the stability constraint come from?

So, we first start with our fully discretised forward Euler scheme. Watch what happens when we apply the forward Euler in time to our spatially discretised equation:

$$u_i^{n+1} = u_i^n + \Delta t \left[ vD \, \frac{u_{i+1}^n - 2u_i^n + u_{i-1}^n}{\Delta r^2} + v(\nu\Sigma_f - \Sigma_a)\, u_i^n \right]$$

where the superscript $n$ denotes the time step and the subscript $i$ denotes the spatial grid point. So, by extension, $u_i^n$ means "the value of $u$ at grid point $i$ after $n$ time steps".

The von Neumann approach (see our LeVeque citation) is to substitute in a trial solution of the form:

$$u_i^n = \xi^n e^{ikr_i}$$

where $\xi$ is what we call the **amplification factor**. What it tells us is by how much a Fourier mode's amplitude changes from one time step to the next. $k$ is the spatial wavenumber of that Fourier mode (we're essentially decomposing our solution into sines and cosines, as one does in Fourier analysis anyway. It's quite neat), and $r_i = i\Delta r$ is the position of grid point $i$. The imaginary $i$ in the exponent (confusingly, not actually the same $i$ as the grid index, Physics is silly and I hate it sometimes) makes this a complex exponential, which is just a way to package our sines and cosines neatly.

So, the logic is this is as follows: if *any* Fourier mode grows without bound (i.e. $|\xi| > 1$), then the whole scheme will blow up as that mode accumulates error across time steps. So, we need $|\xi| \leq 1$ for every possible $k$. This is mostly simple enough in principle.

Let's first substitute. The terms we'll need:

$$u_i^{n+1} = \xi^{n+1} e^{ikr_i} = \xi \cdot \xi^n e^{ikr_i}$$

$$u_i^n = \xi^n e^{ikr_i}$$

$$u_{i+1}^n = \xi^n e^{ikr_{i+1}} = \xi^n e^{ik(r_i + \Delta r)} = \xi^n e^{ikr_i} \cdot e^{ik\Delta r}$$

$$u_{i-1}^n = \xi^n e^{ikr_{i-1}} = \xi^n e^{ik(r_i - \Delta r)} = \xi^n e^{ikr_i} \cdot e^{-ik\Delta r}$$

Now, let's look at ~~Paul Allen's~~ the second derivative term in the diffusion piece, $u_{i+1}^n - 2u_i^n + u_{i-1}^n$:

$$u_{i+1}^n - 2u_i^n + u_{i-1}^n = \xi^n e^{ikr_i}\left( e^{ik\Delta r} - 2 + e^{-ik\Delta r} \right)$$

The bracket here is where some trig magic happens (waves hands). Let me do it in two steps.

**Step 1: Euler's identity.** 

This is where we group the two exponentials:

$$\left(e^{ik\Delta r} + e^{-ik\Delta r}\right) - 2$$

One of the standard forms of Euler's formula is $e^{i\theta} + e^{-i\theta} = 2\cos\theta$. This comes from $e^{i\theta} = \cos\theta + i\sin\theta$ and $e^{-i\theta} = \cos\theta - i\sin\theta$: adding them, the imaginary sine parts cancel and the cosines double up. So with $\theta = k\Delta r$:

$$e^{ik\Delta r} + e^{-ik\Delta r} = 2\cos(k\Delta r)$$

So, by plugging it all back in, we get:

$$\left(e^{ik\Delta r} + e^{-ik\Delta r}\right) - 2 = 2\cos(k\Delta r) - 2 = 2\left[\cos(k\Delta r) - 1\right]$$

Right, onto our next step.

**Step 2: Half-angle identity.** 

So, there's a standard trig identity that comes from the double-angle formula $\cos(2\alpha) = 1 - 2\sin^2\alpha$. Setting $\theta = 2\alpha$ and then rearranging:

$$\cos\theta - 1 = -2\sin^2(\theta/2)$$

With $\theta = k\Delta r$:

$$\cos(k\Delta r) - 1 = -2\sin^2\!\left(\frac{k\Delta r}{2}\right)$$

So:

$$2\left[\cos(k\Delta r) - 1\right] = 2 \cdot \left[-2\sin^2\!\left(\frac{k\Delta r}{2}\right)\right] = -4\sin^2\!\left(\frac{k\Delta r}{2}\right)$$

Let's now put this right back into the second derivative term:

$$u_{i+1}^n - 2u_i^n + u_{i-1}^n = -4\sin^2\!\left(\frac{k\Delta r}{2}\right) \cdot \xi^n e^{ikr_i}$$

You might be wondering why we just did any of that? Well, the reason is that we push all the way to $\sin^2$ rather than stopping at $\cos - 1$. This is because $\sin^2$ is bounded between 0 and 1, which makes the worst-case argument a couple of steps from now really clean.

So, having gotten everything where we need it, we now can substitute everything back into the full forward Euler scheme:

$$\xi \cdot \xi^n e^{ikr_i} = \xi^n e^{ikr_i} + \Delta t \left[ vD \cdot \frac{-4\sin^2(k\Delta r/2)}{\Delta r^2} \cdot \xi^n e^{ikr_i} + v(\nu\Sigma_f - \Sigma_a) \cdot \xi^n e^{ikr_i} \right]$$

The common factor $\xi^n e^{ikr_i}$ appears in every single one of these terms, so we can divide both sides by it and it cleanly cancels:

$$\xi = 1 + \Delta t \left[ vD \cdot \frac{-4\sin^2(k\Delta r/2)}{\Delta r^2} + v(\nu\Sigma_f - \Sigma_a) \right]$$

Hooray!

Now we tidy up the minus sign:

$$\boxed{\xi = 1 - \frac{4vD\Delta t}{\Delta r^2}\sin^2\!\left(\frac{k\Delta r}{2}\right) + v(\nu\Sigma_f - \Sigma_a)\Delta t}$$

And *there's* our amplification factor.

Now, for stability, we need $|\xi| \leq 1$ for every possible wavenumber $k$. Since $\xi$ here is purely real (no imaginary parts survived the cancellation, they were in fact culled. So, everything either cancelled or got absorbed into a $\sin^2$), so this just means:

$$-1 \leq \xi \leq 1$$

The upper bound $\xi \leq 1$ is easy. Our diffusion term is always, always non-positive (since $\sin^2 \geq 0$), and the reactive term $v(\nu\Sigma_f - \Sigma_a)\Delta t$ is positive but small. So $\xi < 1$ quite comfortably, as long as $\Delta t$ is reasonable. Nothing to worry about there.

The lower bound $\xi \geq -1$ is the one that bites us in the rear. That's the constraint that limits $\Delta t$ from above. Let's work it out.

We need:

$$1 - \frac{4vD\Delta t}{\Delta r^2}\sin^2\!\left(\frac{k\Delta r}{2}\right) + v(\nu\Sigma_f - \Sigma_a)\Delta t \geq -1$$

The worst case, i.e. the most negative $\xi$ can get, is when $\sin^2(k\Delta r/2) = 1$. What this corresponds to physically is the shortest-wavelength mode the grid can even represent: one where adjacent grid points oscillate against each other. By plugging that in:

$$1 - \frac{4vD\Delta t}{\Delta r^2} + v(\nu\Sigma_f - \Sigma_a)\Delta t \geq -1$$

Now, we go about rearranging for $\Delta t$:

$$2 \geq \frac{4vD\Delta t}{\Delta r^2} - v(\nu\Sigma_f - \Sigma_a)\Delta t$$

$$2 \geq \Delta t \left[ \frac{4vD}{\Delta r^2} - v(\nu\Sigma_f - \Sigma_a) \right]$$

For our specific problem, the diffusion term $4vD/\Delta r^2$ should utterly dominate the reactive term $v(\nu\Sigma_f - \Sigma_a)$. 

Let's check this: with $v \approx 2 \times 10^9$ cm/s, $D \approx 1$ cm, $\Delta r = 0.05$ cm:

$$\frac{4vD}{\Delta r^2} = \frac{4 \times 2 \times 10^9 \times 1}{0.0025} \approx 3.2 \times 10^{12} \text{ s}^{-1}$$

whereas:

$$v(\nu\Sigma_f - \Sigma_a) = 2 \times 10^9 \times 0.0973 \approx 2 \times 10^8 \text{ s}^{-1}$$

So, the diffusion term is actually about four orders of magnitude larger. If we just drop that reactive term from our  bracket altogether:

$$2 \geq \Delta t \cdot \frac{4vD}{\Delta r^2}$$

$$\Delta t \leq \frac{\Delta r^2}{2vD}$$

And there we have it! This is the stability constraint I waved at earlier. It isn't pulled out of thin air (though I very much wish it was): it's the condition for the worst-case Fourier mode (the one alternating between adjacent grid points) not to amplify each step. Pretty neat when you actually go through it (not so neat for those with Carpal Tunnel).

Right, you can head back to the main notebook now!